# M3: Tuning and Evaluating Model and Agent Performance — 評価パート ハンズオンデモ

本ノートブックは、Google Cloud 認定トレーナー研修「Build Generative AI Applications with Gemini Enterprise Agent Platform」の M3「Tuning and Evaluating Model and Agent Performance」モジュールのうち、**評価（Evaluation）パート**のみを扱うハンズオン教材です。チューニング（SFT / DPO / RFT）パートは、実際のジョブ実行に時間と課金が伴うため本ノートブックには含めていません。

## 生成 AI の評価はなぜ難しいのか

従来のソフトウェアテストは「入力に対して1つの正解」を前提にできますが、生成 AI の出力は自由記述であり、同じ入力でも表現の異なる複数の正解がありえます。そのため、単純な文字列一致だけでは品質を測れず、「意味は合っているが表現が違う」ケースをどう扱うかが評価設計の核心になります。

## Pointwise 評価と Pairwise 評価

**Pointwise 評価**は1つの応答を単独でスコアリングする方式（例:「この応答は 5 点満点中何点か」）です。**Pairwise 評価**は2つの応答を比較して「どちらが優れているか」を判定する方式で、モデルの改善前後の比較（A/Bテスト的な評価）に向いています。本ノートブックでは主に Pointwise 評価を扱います。

## 計算ベース評価とモデルベース評価の違い

**計算ベース評価（count-based / lexicon-based）**は、Accuracy・F1・BLEU・ROUGE のように、正解データ（reference）との一致度を機械的なアルゴリズムで測る方式です。再現性が高く高速ですが、「意味は合っているが表現が違う」応答を不当に低く評価してしまう弱点があります。**モデルベース評価（LLM-as-a-judge）**は、別の LLM を審査役にして応答の品質を判定させる方式です。正解データなしでも使え、意味的な妥当性を評価できますが、審査役の LLM 自体にもブレや偏りがあるため、スコアだけでなく判断理由も確認することが重要です。

## 前提条件

本ノートブックを実行する前に、ターミナルで以下を実行し、Application Default Credentials (ADC) を設定しておいてください。API キーは使用せず、認証は ADC に一本化します。

```bash
gcloud auth application-default login
```

また、リポジトリ直下の `.env` ファイルに `GOOGLE_CLOUD_PROJECT`（対象の Google Cloud プロジェクト ID）と `GOOGLE_CLOUD_LOCATION`（省略時は `global`）を設定しておいてください。


## 評価 API のパッケージについての重要な注記

**評価 API（`client.evals`）はサービスの変化が速い領域です。** 本ノートブックの執筆にあたり実際に SDK をインストールして API を確認したところ、次の点が分かりました。

- モデル呼び出し（`generate_content` など）に使う **`google-genai`** パッケージ（`from google import genai`）の `Client` オブジェクトには、**執筆時点では `evals` 属性は存在しません**（`dir(client.evals)` は `AttributeError` になります）。
- 評価 API は別パッケージの **`google-cloud-aiplatform[evaluation]`**（import 名は `agentplatform`。旧称 `vertexai` も同じ機能を持ちますが、実行すると `FutureWarning: The vertexai.Client class is deprecated. Please use agentplatform.Client instead.` が出ます）に実装されています。
- そのため本ノートブックでは、**通常のモデル呼び出しには `google-genai`（`genai.Client`）、評価には `agentplatform`（`agentplatform.Client`）という2つのクライアントを使い分けます**。どちらも Gemini Enterprise Agent Platform（旧 Vertex AI）上で ADC 認証を使う点は共通です。

もしお使いの環境でメソッド名やシグネチャが異なる場合は、`pip show google-cloud-aiplatform` でバージョンを確認し、`dir(eval_client.evals)` で実際に使えるメソッドを確認しながら読み替えてください。旧 `vertexai.evaluation`（`EvalTask` / `PointwiseMetric`）は後方互換のため残っていますが、`agentplatform.Client` 経由の統合 SDK（`run_inference()` → `evaluate()` → `show()`）が現行の推奨パスです。


## 1. セットアップ

このセクションでは、必要なパッケージのインストールから `.env` の読み込み、2つのクライアント（`genai.Client` と `agentplatform.Client`）の初期化までを行います。以降のすべてのセルで使い回す `MODEL_ID` もここで定義します。


In [ ]:
import sys

# このリポジトリは uv でパッケージを管理しており、uv が作る .venv には
# (uv の設計上) pip 本体がインストールされていません。そのため %pip install や
# !pip install はどちらも "No module named pip" になり失敗します。
# uv 管理の venv では、pip 互換のインターフェースを持つ "uv pip install" を使い、
# --python でこのノートブックのカーネルが使っている Python を明示的に指定して
# インストールします(uv 自体がインストールされていることが前提です)。
!uv pip install --quiet --python {sys.executable} google-genai "google-cloud-aiplatform[evaluation]" pandas scikit-learn matplotlib python-dotenv

# 参考: uv を使わない一般的な Python 環境 (pip が最初から入っている venv など) では、
# 代わりに次の行のコメントを外して実行してください。
# %pip install --upgrade --quiet google-genai "google-cloud-aiplatform[evaluation]" pandas scikit-learn matplotlib python-dotenv


In [ ]:
import os

from dotenv import load_dotenv

# .env ファイルから GOOGLE_GENAI_USE_VERTEXAI / GOOGLE_CLOUD_PROJECT / GOOGLE_CLOUD_LOCATION
# などの環境変数を読み込みます。
load_dotenv()

PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"]
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "global")

print(f"プロジェクト: {PROJECT_ID} / ロケーション: {LOCATION}")


In [ ]:
from google import genai

# 通常のモデル呼び出し (推論) には google-genai の Client を使います。
# vertexai=True を指定することで、Gemini Enterprise Agent Platform (旧 Vertex AI) 経由で
# モデルを呼び出します。認証情報は明示的に渡さず、ADC を利用します。
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

MODEL_ID = "gemini-3.7-flash"

print(f"genai.Client を初期化しました。使用モデル: {MODEL_ID}")


In [ ]:
import agentplatform

# 評価 API (client.evals) は google-genai ではなく google-cloud-aiplatform[evaluation]
# パッケージ (import 名: agentplatform) の Client が提供します。
# 上のセクションで説明した通り、モデル呼び出し用の client とは別のオブジェクトです。
eval_client = agentplatform.Client(project=PROJECT_ID, location=LOCATION)

print("agentplatform.Client (評価用) を初期化しました。")
print("client.evals で使えるメソッド一覧:")
print([m for m in dir(eval_client.evals) if not m.startswith("_")])


## 2. サンプルデータの準備（分類タスク）

このセクションでは、感情分析（ポジティブ／ネガティブ／ニュートラル）のダミーデータを用意します。以降のすべての評価はこのデータセットに対して行います。

誰が読んでも分類が明らかな文章だけを並べると、モデルの正答率が 100% になりやすく、Accuracy・Exact Match・Macro-F1 の値がすべて同じ (= 1.0) になってしまい、指標同士の違いが見えません。そこで、皮肉や複合感情のように**モデルが判断に迷いやすい文章もあえて混ぜています**。これにより、指標ごとに異なる値が出る様子を確認できます。


In [ ]:
import pandas as pd

# 感情分析タスクのダミーデータです。
# text: 分類対象の文章 / reference: 正解ラベル (Positive / Negative / Neutral)
# 実務では自社のデータに置き換えてください。
#
# 各クラス 4 件ずつのうち、後半 2 件はあえて判断が分かれやすい紛らわしい文章
# (皮肉、一部だけ好意的な表現を含む不満、弱い肯定/中立の境界など) にしています。
# Gemini 3.7-flash で実際に試したところ、これらは安定して誤分類されるため、
# Accuracy / Exact Match / Macro-F1 が 1.0 に張り付かず、指標同士の違いを確認できます。
eval_df = pd.DataFrame({
    "text": [
        # Positive (分類しやすい文章)
        "この製品は期待以上の品質で、大満足です。",
        "毎日使うのが楽しみになるくらい気に入っています。",
        "スタッフの対応がとても丁寧で感激しました。",
        "とても静かで拍子抜けするくらい快適でした。",
        # Negative (前半2件は分類しやすく、後半2件は紛らわしい)
        "サポート対応が遅く、二度と利用したくありません。",
        "配送されてきた商品が破損していて非常に残念でした。",
        "画質は綺麗ですが、バッテリーの持ちが悪いです。",
        "サポートセンターにつながるまで1時間待ちましたが、対応自体はとても丁寧でした。",
        # Neutral (前半2件は分類しやすく、後半2件は紛らわしい)
        "会議は予定通り15時から始まりました。",
        "本日の会議室は3階の第2会議室です。",
        "値段の割にはまずまずの出来だと思います。",
        "別に悪くはないけど、また使うかと言われると微妙です。",
    ],
    "reference": [
        "Positive", "Positive", "Positive", "Positive",
        "Negative", "Negative", "Negative", "Negative",
        "Neutral", "Neutral", "Neutral", "Neutral",
    ],
})

print(f"サンプル件数: {len(eval_df)} 件")
eval_df


## 3. 推論の実行（response 列を作る）

このセクションでは、用意した文章それぞれについてモデルに分類させ、`response` 列を作ります。まず `client.evals.run_inference()` を使う書き方を試し、SDK バージョンの違いなどでうまく動かない場合は、`google-genai` の `generate_content()` を使った手動ループにフォールバックします。


In [ ]:
CLASSIFY_INSTRUCTION = (
    "次の文章の感情を Positive、Negative、Neutral のいずれか一語だけで分類してください。"
    "説明や理由、記号は一切含めず、ラベル一語のみを出力してください。\n\n"
    "文章: {text}"
)

# run_inference() に渡す DataFrame は "prompt" 列 (モデルへの入力全文) を必要とします。
# text (分類対象の文) と reference (正解ラベル) を保持したまま、
# 分類指示を組み込んだ prompt 列を作成します。
inference_input_df = eval_df.copy()
inference_input_df["prompt"] = inference_input_df["text"].apply(
    lambda t: CLASSIFY_INSTRUCTION.format(text=t)
)

try:
    inference_result = eval_client.evals.run_inference(
        model=MODEL_ID,
        src=inference_input_df[["prompt", "reference"]],
    )
    # run_inference() の戻り値 (EvaluationDataset) は eval_dataset_df 属性で
    # pandas DataFrame として取り出せます。
    result_df = (
        inference_result.eval_dataset_df
        if hasattr(inference_result, "eval_dataset_df")
        else inference_result
    )
    # run_inference() には prompt/reference 列しか渡していないため、text (元の文章) や
    # reference が結果側に含まれない SDK バージョンに備えて、念のため明示的に
    # 元データから引き直します。
    if "reference" not in result_df.columns:
        result_df["reference"] = inference_input_df["reference"].values
    if "text" not in result_df.columns:
        result_df["text"] = inference_input_df["text"].values
    print("client.evals.run_inference() を使って推論を実行しました。")
except Exception as e:
    # SDK バージョン差異などで run_inference() が使えない場合は、
    # generate_content() を使った手動ループにフォールバックします。
    print(f"client.evals.run_inference() が失敗したため、手動ループにフォールバックします: {type(e).__name__}: {e}")
    responses = []
    for prompt in inference_input_df["prompt"]:
        resp = client.models.generate_content(model=MODEL_ID, contents=prompt)
        responses.append(resp.text.strip())
    result_df = inference_input_df.copy()
    result_df["response"] = responses

result_df[["text", "reference", "response"]]


## 4. 計算ベース評価（Accuracy / Exact Match / Macro-F1）

`response` と `reference` を突き合わせて、**Accuracy**、**Exact Match**、**Macro-F1** を計算します。これが M3 資料でいう計算ベース評価（count-based / lexicon-based）です。


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

y_true = result_df["reference"].tolist()
y_pred = result_df["response"].tolist()

# Accuracy: 全件中、正解ラベルと一致した割合。
accuracy = accuracy_score(y_true, y_pred)

# Exact Match: この分類タスクでは応答が単一ラベルのみのため、Accuracy と同じ計算になります。
# (自由記述の要約・翻訳タスクなどでは、文字列単位の完全一致を測る指標として Accuracy と区別されます。)
exact_match = sum(1 for t, p in zip(y_true, y_pred) if t == p) / len(y_true)

# Macro-F1: クラスごとに F1 を計算し、単純平均したもの。少数派クラスの性能低下も
# 見逃さずに反映されるため、クラスの件数に偏りがあるデータセットで重視されます。
macro_f1 = f1_score(y_true, y_pred, average="macro", labels=["Positive", "Negative", "Neutral"])

print(f"Accuracy   : {accuracy:.3f}")
print(f"Exact Match: {exact_match:.3f}")
print(f"Macro-F1   : {macro_f1:.3f}")


続けて、`client.evals.evaluate()` に相当する評価 API でも Exact Match 相当の指標（`exact_match`）を算出できるか試します。scikit-learn の素朴な実装と評価 API の結果を突き合わせることで、どちらも同じものを測っていることを確認します。


In [ ]:
from agentplatform import types

try:
    exact_match_result = eval_client.evals.evaluate(
        dataset=result_df[["prompt", "response", "reference"]],
        metrics=[types.Metric(name="exact_match")],
    )
    exact_match_result.show()

    sdk_scores = [
        case.response_candidate_results[0].metric_results["exact_match"].score
        for case in exact_match_result.eval_case_results
    ]
    sdk_exact_match = sum(sdk_scores) / len(sdk_scores)
    print(f"scikit-learn で計算した Exact Match : {exact_match:.3f}")
    print(f"評価 API で計算した Exact Match     : {sdk_exact_match:.3f}")
except Exception as e:
    # SDK バージョン差異でメトリック名やシグネチャが異なる場合はここで失敗します。
    # その場合は上のセルの scikit-learn 実装の結果を正としてください。
    print(f"評価 API での算出に失敗しました (scikit-learn の結果を使用してください): {type(e).__name__}: {e}")


### Macro-F1 と Micro-F1 の違い

Macro-F1 はクラスごとの F1 を単純平均するため、件数の少ないクラスも均等に扱います。一方 Micro-F1 は全件をまとめて TP/FP/FN を集計してから F1 を計算するため、件数の多いクラスの性能に結果が引っ張られます。クラスごとの件数をあえて不均衡にしたミニデータセットで、両者の数値が変わる様子を確認します。


In [ ]:
# クラスごとの件数をあえて不均衡にしたミニデータセット
# (Positive が多数派、Negative と Neutral が少数派で、少数派クラスの予測を外した想定)
imbalanced_true = ["Positive"] * 8 + ["Negative"] * 2 + ["Neutral"] * 2
imbalanced_pred = ["Positive"] * 8 + ["Positive"] * 2 + ["Positive"] * 2  # 少数派を全て誤分類

macro = f1_score(
    imbalanced_true, imbalanced_pred, average="macro",
    labels=["Positive", "Negative", "Neutral"], zero_division=0,
)
micro = f1_score(
    imbalanced_true, imbalanced_pred, average="micro",
    labels=["Positive", "Negative", "Neutral"], zero_division=0,
)

# - Micro-F1 = 0.667 — 全12件をまとめて集計するので「8/12 = 67%当たっている」となり、そこそこ良く見えます
# - Macro-F1 = 0.267 — クラスごとにF1を出して平均します。PositiveのF1は0.8程度ですが、NegativeとNeutralのF1は共に0（1件も当てられていない）。この3つを平均するので大きく下がります

print(f"Macro-F1: {macro:.3f}  <- 少数派クラス (Negative, Neutral) の全滅が反映され、大きく下がります")
print(f"Micro-F1: {micro:.3f}  <- 多数派クラス (Positive) が全問正解のため、高めの値になります")


### 補足: 自由記述タスクでは Exact Match が実態に合わないことがある

ここまでの感情分析タスクでは、モデルの応答が Positive/Negative/Neutral のいずれか一語に限定されていたため、**Accuracy と Exact Match は常に同じ値**になりました。しかし、要約や翻訳のような自由記述タスクでは応答が自由な文字列になるため、「意味は完全に正しいのに、模範解答と1文字でも違うだけで Exact Match は 0 点になる」というケースが自然に発生します。日本語→英語の翻訳タスクを例に、この違いを実際に確認します。


In [ ]:
from google.genai import types as genai_types

TRANSLATE_INSTRUCTION = (
    "次の日本語の文章を、自然な英語に翻訳してください。"
    "説明や前置きは一切含めず、翻訳結果の英文のみを出力してください。\n\n"
    "文章: {text}"
)

# 日本語の文章と、あらかじめ用意した模範訳 (reference) です。
# 実際に翻訳させると、意味は正しいのに模範解答と一字一句までは一致しないケース
# ("It's" と "It is"、"3 PM" と "3:00 PM" のようなつづり・言い回しの違い) が
# 自然に発生します。
translation_df = pd.DataFrame({
    "text": [
        "今日の天気は晴れです。",
        "会議は午後3時に始まります。",
        "この本はとても面白いです。",
        "駅までは歩いて10分です。",
        "私は毎朝コーヒーを飲みます。",
        "明日の会議はキャンセルになりました。",
        "この製品は日本製です。",
        "彼女は3年前に日本に来ました。",
    ],
    "reference": [
        "The weather is sunny today.",
        "The meeting starts at 3 PM.",
        "This book is very interesting.",
        "It's a 10-minute walk to the station.",
        "I drink coffee every morning.",
        "Tomorrow's meeting has been cancelled.",
        "This product is made in Japan.",
        "She came to Japan three years ago.",
    ],
})

# temperature=0 にして、何度実行しても翻訳結果が変わらないようにします。
deterministic_config = genai_types.GenerateContentConfig(temperature=0)

translation_df["prompt"] = translation_df["text"].apply(
    lambda t: TRANSLATE_INSTRUCTION.format(text=t)
)
translation_df["response"] = [
    client.models.generate_content(
        model=MODEL_ID, contents=p, config=deterministic_config
    ).text.strip()
    for p in translation_df["prompt"]
]

translation_df[["text", "reference", "response"]]


In [ ]:
from agentplatform import types

# Exact Match: 完全一致率。自由記述タスクでは表現の揺れに弱く、
# 意味的に正しくても不一致 (0点) になりがちです。
translation_exact_match = sum(
    1 for r, p in zip(translation_df["reference"], translation_df["response"]) if r == p
) / len(translation_df)
print(f"Exact Match: {translation_exact_match:.3f}")

# ROUGE-L: 参照文と応答文のあいだの最長共通部分列 (語の並びの重なり) をもとにした
# スコアで、表現が多少違っても意味的に近ければ高い値になります。
# client.evals.evaluate() の rouge_l_sum メトリックで、ケースごとに計算します。
try:
    rouge_result = eval_client.evals.evaluate(
        dataset=translation_df[["prompt", "response", "reference"]],
        metrics=[types.Metric(name="exact_match"), types.Metric(name="rouge_l_sum")],
    )
    per_case_scores = []
    for i, case in enumerate(rouge_result.eval_case_results):
        for candidate in case.response_candidate_results:
            per_case_scores.append({
                "reference": translation_df.iloc[i]["reference"],
                "response": translation_df.iloc[i]["response"],
                "exact_match": candidate.metric_results["exact_match"].score,
                "rouge_l_sum": candidate.metric_results["rouge_l_sum"].score,
            })
    per_case_df = pd.DataFrame(per_case_scores)
    print(f"平均 ROUGE-L    : {per_case_df['rouge_l_sum'].mean():.3f}")
    per_case_df
except Exception as e:
    print(f"評価 API での算出に失敗しました: {type(e).__name__}: {e}")


Exact Match は 1.0 に届かない一方（"3:00 PM" vs "3 PM"、"It is" vs "It's" のような完全に正しい訳を不正解扱いしてしまうため）、ROUGE-L はこれらのケースにも高いスコア (0.9 前後) を与えます。これが、ノートブック冒頭で説明した「計算ベース評価は意味は合っているが表現が違う応答を不当に低く評価してしまう」という弱点の具体例です。**分類タスクでは Accuracy / Exact Match が有効ですが、自由記述タスクでは ROUGE や BLEU のような部分一致を評価できる指標を使う方が実態に合います。**（`temperature=0` にしていても、モデルのバージョン更新などで実行のたびに Exact Match の具体的な値は変わりえます。)


## 5. モデルベース評価（LLM as a judge）

このセクションでは、`agentplatform` の定義済みルーブリック指標（`RubricMetric`）を使い、審査役の LLM に応答の品質を採点させます。**正解データ (reference) が一切なくても使える**のがこの方式の最大の利点です。

### 評価対象は「自由記述の回答」にします

セクション2〜4で使ってきた感情分析の応答は `Positive` / `Negative` / `Neutral` という**一語のラベル**でした。これを品質指標にかけても、指示どおり一語で答えている以上すべて満点になり、スコアに差が出ません（実際に試すと全件 1.0 になり、評価として何も語らない結果になります）。

モデルベース評価の本領は、**正解が一つに定まらない自由記述の質**を測れる点にあります。そこでこのセクションでは、同じ質問に対して品質の異なる4つの回答を用意し、スコアが実際に割れる様子を確認します。

### 2つの指標の違い

| 指標 | 測るもの |
| --- | --- |
| `general_quality` | 回答の**総合的な品質**。質問にきちんと答えられているか、内容が妥当か |
| `text_quality` | **文章としての品質**。流暢さ、一貫性、分かりやすさ |

この違いは「D: 質問に答えていない」の回答で顕著に表れます。D は日本語として整っているものの質問には答えていないため、`text_quality` では比較的高い点が付く一方、`general_quality` では大きく減点される傾向があります。**指標の選び方を誤ると「流暢に的外れなことを言うモデル」を高評価してしまう**という、実務上の重要な教訓です。

### スコアは実行のたびに変動します

審査役 LLM は評価のたびに「その回答が満たすべき観点（ルーブリック）」を自動生成するため、**同じデータでも実行ごとにスコアが変わります**。実際に筆者が `general_quality` で4回試したところ、次のように結果が動きました。

| 回答 | 1回目 | 2回目 | 3回目 | 4回目 |
| --- | --- | --- | --- | --- |
| A: 高品質 | 1.00 | 0.78 | 1.00 | 0.88 |
| B: 中品質 | 1.00 | 1.00 | 0.86 | 0.67 |
| C: 低品質 | 0.14 | 0.20 | 0.17 | 0.43 |
| D: 質問に答えていない | 0.25 | 0.60 | 0.71 | 0.14 |

「情報量ゼロの C が低スコアになる」という大枠は概ね安定していますが、**A と B の順位は入れ替わることがあり、D にいたっては 0.14〜0.71 と大きく振れています**。

ここから読み取るべきは、モデルベース評価のスコアは絶対的な真値ではなく**目安**だということです。0.88 と 1.00 のような僅差を「有意な品質差」と解釈してはいけません。実務では、複数回実行して平均を取る（`judge_model_sampling_count` の指定）、あるいは人手評価と突き合わせて審査役自体の信頼性を検証する、といった対策を取ります。

なお審査役 LLM の判定は数%程度の確率でまれに失敗し（`Rubric results could not be reliably computed` など）、実行中に大量のエラーログが表示されることがあります。これも審査役 LLM のブレの一例です。`evaluate()` 自体は停止せず処理を継続するので、慌てず結果を確認してください。


In [ ]:
from agentplatform import types

# モデルベース評価のデモ用に、同じ質問に対して「品質の異なる4つの回答」を用意します。
# 一語のラベルではなく自由記述にすることで、品質スコアに差が出る様子を確認できます。
# 実務では、自社のアプリが実際に生成した回答をここに並べてください。
QUESTION = "生成AIアプリケーションの評価がなぜ難しいのか、初心者にもわかるように説明してください。"

quality_demo_df = pd.DataFrame({
    "品質": ["A: 高品質", "B: 中品質", "C: 低品質", "D: 質問に答えていない"],
    "prompt": [QUESTION] * 4,
    "response": [
        # A: 理由を構造的に整理し、具体的に説明した回答
        "生成AIの評価が難しい理由は主に3つあります。第一に、出力が自由記述であるため「正解が一つに定まらない」点です。"
        "同じ質問でも表現の異なる複数の正しい回答がありえます。第二に、同じ入力でも実行のたびに出力が変わりうる非決定性があります。"
        "第三に、事実誤り・論理の飛躍・語調の不適切さなど、評価すべき観点が多面的である点です。"
        "そのため単純な文字列一致では品質を測れず、計算ベース指標とLLMによる判定を組み合わせる必要があります。",
        # B: 内容は正しいが浅い回答
        "出力が毎回変わることがあり、正解も一つではないため、単純な比較では評価しづらいからです。",
        # C: 情報量がほぼゼロの回答
        "難しいです。",
        # D: 流暢だが質問に答えていない回答
        "生成AIは近年急速に普及しており、多くの企業が導入を進めています。市場規模は今後も拡大する見込みです。",
    ],
})

quality_demo_df[["品質", "response"]]


In [ ]:
# GENERAL_QUALITY: 回答の総合的な品質 (質問に答えているか、内容が妥当かなど)
# TEXT_QUALITY  : 文章としての品質 (流暢さ・一貫性・分かりやすさなど)
# いずれも reference (正解データ) を渡していない点に注目してください。
try:
    quality_result = eval_client.evals.evaluate(
        dataset=quality_demo_df[["prompt", "response"]],
        metrics=[types.RubricMetric.GENERAL_QUALITY, types.RubricMetric.TEXT_QUALITY],
    )
    quality_result.show()

    # --- スコア一覧を表にまとめます ---------------------------------------
    rows = []
    for i, case in enumerate(quality_result.eval_case_results):
        row = {"品質": quality_demo_df.iloc[i]["品質"]}
        for candidate in case.response_candidate_results:
            for metric_name, metric_result in candidate.metric_results.items():
                # 判定に失敗したケースは score が None になります。
                row[metric_name] = metric_result.score
        rows.append(row)
    display(pd.DataFrame(rows))
    print("※ 審査役 LLM は評価のたびに観点を生成し直すため、実行ごとにスコアは変動します。")
except Exception as e:
    print(f"モデルベース評価の実行に失敗しました: {type(e).__name__}: {e}")
    print("dir(types.RubricMetric) で使用可能な指標名を確認してください。")


### スコアの内訳を確認する

上の表は「どの回答が高評価か」しか教えてくれません。**なぜそのスコアになったのか**を確認します。

定義済みメトリックは `explanation` が空で返り、代わりに `rubric_verdicts` に判断根拠が格納されています。ここには審査役 LLM がその場で自動生成した「観点」と、それぞれの合否・理由が入っています。

そして **スコアは「合格した観点数 ÷ 生成された観点の総数」で計算されます**。下のセルで実際に検算してみてください。

注意点が2つあります。

- **観点の数と内容は回答ごとに異なります。** 全員が同じテストを受けているわけではなく、一人ひとり別の問題用紙で採点されているイメージです。これが実行ごとにスコアが変動する根本原因です。
- **観点は英語で生成されます。** 日本語で質問しても審査役は英語で観点を組み立てるため、ここは制御できません。


In [ ]:
# 低品質な回答 (C) について、審査役がどの観点で合格/不合格を出したかを表で確認します。
# 「合格した観点数 ÷ 全観点数」が上の表のスコアと一致することを確かめてください。
target_index = 2  # C: 低品質。0=A, 1=B, 3=D に変えて他の回答も見てみてください。

print(f"=== 「{quality_demo_df.iloc[target_index]['品質']}」の採点内訳 ===")
print(f"回答: {quality_demo_df.iloc[target_index]['response']}")

target_case = quality_result.eval_case_results[target_index]
for candidate in target_case.response_candidate_results:
    for metric_name, metric_result in candidate.metric_results.items():
        if metric_result.score is None:
            print(f"\n[{metric_name}] スコア取得エラー: {metric_result.error_message}")
            continue

        verdicts = metric_result.rubric_verdicts or []
        passed = sum(1 for v in verdicts if v.verdict is True)
        print(f"\n[{metric_name}] score={metric_result.score:.3f}  (合格 {passed} / 全 {len(verdicts)} 観点)")

        verdict_df = pd.DataFrame([
            {
                "判定": "合格" if v.verdict is True else "不合格",
                # 観点の記述に審査役の思考過程 (STEP 1: ...) が続くことがあるため 1 行目だけ使います。
                "観点": v.evaluated_rubric.content.property.description.splitlines()[0].strip(),
            }
            for v in verdicts
        ])
        display(verdict_df)


## 6. カスタムルーブリックによる独自指標

**何を評価するか（criteria）**と**どう採点するか（rating rubric）**を自分で定義し、独自の評価基準を作ります。ここでは、この Notebook のテーマ（感情分析の分類結果）に合わせて、「回答がラベル一語のみで簡潔か」「絵文字を使っていないか」を判定するカスタム指標を作成します。

実装上、次の2点がハマりどころです。

- **`judge_model` には完全修飾リソース名が必要です。** `"gemini-3.7-flash"` のような短いモデル ID をそのまま渡すと `Invalid autorater model resource name` エラーになります。`projects/{PROJECT_ID}/locations/{LOCATION}/publishers/google/models/{MODEL_ID}` の形式で指定してください。
- **`types.MetricPromptBuilder` で `criteria` / `rating_scores` を組み立てる書き方は、審査役 LLM が JSON ではなく Markdown で応答してしまい `Error parsing JSON` で失敗することが多くありました。** そのため本ノートブックでは、`prompt_template` に採点基準と出力形式を直接書き、`result_parsing_function` で応答から JSON を自分で取り出す方式を採用しています。考え方（基準＋ルーブリックを与えて LLM に採点させる）は同じです。


In [ ]:
from agentplatform import types

# judge_model には、モデル ID をそのまま渡すのではなく「完全修飾リソース名」が必要です。
# "gemini-3.7-flash" のような短い ID を渡すと、SDK はその値を無加工で
# autorater_model フィールドに入れて送信するため、サーバー側で
#   400 INVALID_ARGUMENT ... Invalid autorater model resource name
# となって失敗します。
JUDGE_MODEL = (
    f"projects/{PROJECT_ID}/locations/{LOCATION}/publishers/google/models/{MODEL_ID}"
)

# criteria と rating_scores を types.MetricPromptBuilder で組み立てる書き方は
# 公式ドキュメントにも載っていますが、実際に試したところ、審査役 LLM が期待される
# JSON 形式ではなく Markdown 形式 ("**Rating:** 5" など) で応答してしまい、
# サーバー側のパースが "Error parsing JSON" で失敗するケースが多くありました。
# そこで本ノートブックでは、より確実に動作する方式として
#   (1) prompt_template に「基準 + 採点ルーブリック + 出力形式」を直接書く
#   (2) result_parsing_function で応答から JSON を自分で取り出す
# という組み合わせを使います (Google 公式サンプルでも使われている書き方です)。
conciseness_metric = types.LLMMetric(
    name="label_conciseness_check",
    prompt_template="""以下のプロンプトと応答を確認し、応答が分類ラベルの出力ルールを守れているかを判定してください。

基準1 (conciseness): 応答が Positive、Negative、Neutral のいずれか一語のみで構成されており、説明文や理由、句読点などの余計な文字列を含んでいないこと。
基準2 (no_emoji): 応答に絵文字が一切含まれていないこと。

採点ルーブリック:
  score = 1: 両方の基準を満たしている (簡潔で、絵文字も含まれていない)
  score = 0: いずれか一方でも基準を満たしていない

prompt: {prompt}
response: {response}

次の形式の JSON オブジェクトのみを返してください (前後に説明文を付けないでください):
{"score": 1, "explanation": "そう判断した理由"}
""",
    # result_parsing_function は「文字列として渡した Python コード」です。
    # 審査役 LLM の生の応答 (responses) を受け取り、score と explanation を持つ
    # 辞書を返す parse_results 関数を定義します。
    result_parsing_function="""
import json, re
def parse_results(responses):
    text = responses[0]
    m = re.search(r'\\{.*\\}', text, re.DOTALL)
    if not m:
        return {"score": 0.0, "explanation": "JSON が見つかりませんでした: " + text[:200]}
    try:
        d = json.loads(m.group(0))
        return {"score": float(d.get("score", 0)), "explanation": str(d.get("explanation", ""))}
    except Exception as e:
        return {"score": 0.0, "explanation": "パースに失敗しました: " + str(e)}
""",
    judge_model=JUDGE_MODEL,
)

try:
    custom_result = eval_client.evals.evaluate(
        dataset=result_df[["prompt", "response"]],
        metrics=[conciseness_metric],
    )
    custom_result.show()

    # ケースごとのスコアと判断理由を確認します。
    for i, case in enumerate(custom_result.eval_case_results):
        for candidate in case.response_candidate_results:
            for metric_name, metric_result in candidate.metric_results.items():
                if metric_result.score is None:
                    print(f"[case {i}] スコア取得エラー: {metric_result.error_message}")
                else:
                    print(
                        f"[case {i}] response={result_df.iloc[i]['response']!r} "
                        f"score={metric_result.score}"
                    )
                    print(f"  explanation: {metric_result.explanation}")
except Exception as e:
    # 評価 API 自体が使えない環境では、「基準 + 採点ルーブリックをプロンプトに埋め込んで
    # LLM に判定させる」という考え方だけを、素朴な generate_content() 呼び出しで再現します。
    print(f"評価 API 経由のカスタム指標に失敗しました。素朴な実装にフォールバックします: {type(e).__name__}: {e}")

    judge_prompt_template = (
        "あなたは採点者です。以下の基準に従って、prompt に対する response を判定してください。\n"
        "基準1 (conciseness): 応答が Positive/Negative/Neutral のいずれか一語のみであること。\n"
        "基準2 (no_emoji): 応答に絵文字が含まれていないこと。\n"
        "両方満たしていれば 1、どちらか満たしていなければ 0 だけを出力してください。\n\n"
        "prompt: {prompt}\nresponse: {response}"
    )
    scores = []
    for _, row in result_df.head(5).iterrows():
        judge_prompt = judge_prompt_template.format(prompt=row["prompt"], response=row["response"])
        judge_resp = client.models.generate_content(model=MODEL_ID, contents=judge_prompt)
        scores.append(judge_resp.text.strip())
    print(scores)


## 7. 結果の可視化・比較

ここまでで計算した計算ベース指標とモデルベース指標の結果を1つの表にまとめます。


In [ ]:
summary_df = pd.DataFrame({
    "指標": ["Accuracy", "Exact Match", "Macro-F1"],
    "種別": ["計算ベース", "計算ベース", "計算ベース"],
    "スコア": [accuracy, exact_match, macro_f1],
})

summary_df


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# matplotlib の既定フォント (DejaVu Sans) には日本語のグリフが含まれていません。
# そのままグラフに日本語を描画すると
#   UserWarning: Glyph 12473 (\N{KATAKANA LETTER SU}) missing from font(s) DejaVu Sans
# のような警告が大量に出て、文字が豆腐 (□) になります。
# 環境ごとに入っているフォントが異なるため、候補の中から実際に利用できるものを
# 自動で選んで設定します。
JP_FONT_CANDIDATES = [
    "Hiragino Sans",              # macOS
    "Hiragino Kaku Gothic ProN",  # macOS
    "YuGothic",                   # macOS
    "Yu Gothic",                  # Windows
    "Meiryo",                     # Windows
    "MS Gothic",                  # Windows
    "Noto Sans CJK JP",           # Linux (fonts-noto-cjk)
    "Noto Sans JP",
    "IPAexGothic",                # Linux (fonts-ipaexfont)
    "IPAGothic",
    "TakaoGothic",
    "Arial Unicode MS",
]
available_fonts = {f.name for f in fm.fontManager.ttflist}
jp_font = next((f for f in JP_FONT_CANDIDATES if f in available_fonts), None)

if jp_font:
    plt.rcParams["font.family"] = jp_font
    # 日本語フォントに切り替えると軸のマイナス記号が豆腐になることがあるため、
    # ASCII のハイフンを使う設定にしておきます。
    plt.rcParams["axes.unicode_minus"] = False
    print(f"グラフ用の日本語フォント: {jp_font}")
else:
    # 日本語フォントが1つも見つからない環境では、ラベルを英語にして警告を回避します。
    print("日本語フォントが見つからないため、グラフのラベルは英語で描画します。")

try:
    # client.evals.show() に相当する評価 API 側の可視化がある場合はそちらも参考にしてください
    # (上のセクションの result.show() が該当します)。ここでは matplotlib で簡単な棒グラフを作ります。
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(summary_df["指標"], summary_df["スコア"], color=["#4C72B0", "#55A868", "#C44E52"])
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("スコア" if jp_font else "Score")
    ax.set_title("計算ベース評価指標の比較" if jp_font else "Computation-based metrics")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"グラフの描画に失敗しました: {type(e).__name__}: {e}")


## 8. LLM アプリのユニットテスト

資料の分類タスクの例にならい、`unittest` で分類結果を検証するテストケースを書きます。あえて1件は失敗するテストケースを混ぜています。テスト失敗は「モデルが間違っているのか、テスト側の期待値が甘いのか」を見極めるサインになる、という資料の指摘を確認してください。


In [ ]:
import unittest


def classify_sentiment(text: str) -> str:
    prompt = CLASSIFY_INSTRUCTION.format(text=text)
    response = client.models.generate_content(model=MODEL_ID, contents=prompt)
    return response.text.strip()


class TestSentimentClassification(unittest.TestCase):
    def test_positive_case(self):
        result = classify_sentiment("この製品は期待以上の品質で、大満足です。")
        self.assertEqual(result, "Positive")

    def test_negative_case(self):
        result = classify_sentiment("サポート対応が遅く、二度と利用したくありません。")
        self.assertEqual(result, "Negative")

    def test_neutral_case_intentionally_too_strict(self):
        # このテストはあえて厳しすぎる期待値 ("Neutral。") にしています。
        # モデルは "Neutral" とだけ返すよう指示されているため、実際には一致せず失敗します。
        # -> このテストの失敗は「モデルが間違っている」のではなく「テスト側の期待値が甘い
        #    (句読点まで期待している)」ことが原因です。テスト失敗を見たら、まずどちらが
        #    原因かを切り分ける必要がある、という M3 資料の指摘そのものです。
        result = classify_sentiment("会議は予定通り15時から始まりました。")
        self.assertEqual(result, "Neutral。")


# unittest.main() は Jupyter のデフォルト引数と衝突するため、
# 明示的に argv を指定して実行します。
suite = unittest.TestLoader().loadTestsFromTestCase(TestSentimentClassification)
unittest.TextTestRunner(verbosity=2).run(suite)


## 9. まとめ

| 評価方法 | 正解データ | 得意なこと | 苦手なこと |
| --- | --- | --- | --- |
| 計算ベース評価 (Accuracy / F1 / Exact Match など) | 必須 | 高速・再現性が高い・コストがかからない | 表現の違いを不当に減点しやすい |
| モデルベース評価 (LLM as a judge) | 不要 | 意味的な妥当性・流暢さなど自由記述の質を評価できる | 審査役 LLM 自体のブレや偏りが結果に混ざる |
| ユニットテスト (`unittest` など) | 必須 (期待値として) | 「守るべき最低限の仕様」を明確に固定できる | 期待値が厳しすぎる/緩すぎると誤ったシグナルになる |

今回は評価パートのみを扱いました。実際にモデルをチューニングする（SFT など）場合は、本ノートブックで計算した評価結果を「チューニング前後でどれだけ改善したか」を測る指標として使うことになります。
